# Extract Data

### 1. Accessing Flight Data

Connecting to the database,

In [1]:
# Importing,
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer

# Creating the engine,
engine = create_engine("sqlite:///flight_data.db", echo=False)
connection = engine.connect()
metadata = MetaData()

# Creating table,
flights = Table(
    "flights",
    metadata,
    Column("FL_DATE", String),
    Column("OP_CARRIER", String),
    Column("OP_CARRIER_FL_NUM", Integer),
    Column("ORIGIN", String),
    Column("DEST", String),
    Column("CRS_DEP_TIME", String),
    Column("DEP_TIME", String),
    Column("DEP_DELAY", Integer),
    Column("TAXI_OUT", Integer),
    Column("WHEELS_OFF", String),
    Column("WHEELS_ON", String),
    Column("TAXI_IN", Integer),
    Column("CRS_ARR_TIME", String),
    Column("ARR_TIME", String),
    Column("ARR_DELAY", Integer),
    Column("CANCELLED", Integer),
    Column("DIVERTED", Integer),
    Column("CRS_ELAPSED_TIME", Integer),
    Column("ACTUAL_ELAPSED_TIME", Integer),
    Column("AIR_TIME", Integer),
    Column("DISTANCE", Integer),
    Column("TOTAL_DELAY", Integer)
)

Creating the select statement,

In [2]:
select_statement = flights.select().where((flights.c.ORIGIN == "JFK") & (flights.c.DEP_DELAY > 0))
result = connection.execute(select_statement)

The resulting rows represent delayed flights that only originated from JFK airport. To double check, we explicitly print a few of them out,

In [3]:
preview_rows = connection.execute(select_statement.limit(10)).mappings()
for row in preview_rows:
    print(row)

{'FL_DATE': '2018-01-01', 'OP_CARRIER': 'AS', 'OP_CARRIER_FL_NUM': 7, 'ORIGIN': 'JFK', 'DEST': 'SEA', 'CRS_DEP_TIME': '07:15', 'DEP_TIME': '07:19', 'DEP_DELAY': 4, 'TAXI_OUT': 18, 'WHEELS_OFF': '07:37', 'WHEELS_ON': '10:08', 'TAXI_IN': 16, 'CRS_ARR_TIME': '10:45', 'ARR_TIME': '10:24', 'ARR_DELAY': -21, 'CANCELLED': 0, 'DIVERTED': 0, 'CRS_ELAPSED_TIME': 390, 'ACTUAL_ELAPSED_TIME': 365, 'AIR_TIME': 331, 'DISTANCE': 2422, 'TOTAL_DELAY': -25}
{'FL_DATE': '2018-01-01', 'OP_CARRIER': '9E', 'OP_CARRIER_FL_NUM': 4001, 'ORIGIN': 'JFK', 'DEST': 'CLE', 'CRS_DEP_TIME': '14:55', 'DEP_TIME': '14:58', 'DEP_DELAY': 3, 'TAXI_OUT': 27, 'WHEELS_OFF': '15:25', 'WHEELS_ON': '16:45', 'TAXI_IN': 7, 'CRS_ARR_TIME': '17:10', 'ARR_TIME': '16:52', 'ARR_DELAY': -18, 'CANCELLED': 0, 'DIVERTED': 0, 'CRS_ELAPSED_TIME': 135, 'ACTUAL_ELAPSED_TIME': 114, 'AIR_TIME': 80, 'DISTANCE': 425, 'TOTAL_DELAY': -21}
{'FL_DATE': '2018-01-01', 'OP_CARRIER': '9E', 'OP_CARRIER_FL_NUM': 4002, 'ORIGIN': 'JFK', 'DEST': 'DTW', 'CRS_DEP_

### 2. Creating New Database

We need a new engine to create the database,

In [4]:
# Creating the engine,
second_engine = create_engine("sqlite:///jfk_delayed_flights.db", echo=False)
second_metadata = MetaData()

# Creating table,
delayed_flights = Table(
    "delayed_flights",
    second_metadata,
    Column("FL_DATE", String),
    Column("OP_CARRIER", String),
    Column("OP_CARRIER_FL_NUM", Integer),
    Column("ORIGIN", String),
    Column("DEST", String),
    Column("CRS_DEP_TIME", String),
    Column("DEP_TIME", String),
    Column("DEP_DELAY", Integer),
    Column("TAXI_OUT", Integer),
    Column("WHEELS_OFF", String),
    Column("WHEELS_ON", String),
    Column("TAXI_IN", Integer),
    Column("CRS_ARR_TIME", String),
    Column("ARR_TIME", String),
    Column("ARR_DELAY", Integer),
    Column("CANCELLED", Integer),
    Column("DIVERTED", Integer),
    Column("CRS_ELAPSED_TIME", Integer),
    Column("ACTUAL_ELAPSED_TIME", Integer),
    Column("AIR_TIME", Integer),
    Column("DISTANCE", Integer),
    Column("TOTAL_DELAY", Integer)
)

second_metadata.create_all(second_engine)

Now we insert the data in batches,

In [5]:
BATCH_SIZE = 5000
selected_rows = connection.execute(select_statement).mappings()
insert_statement = delayed_flights.insert()
rows_inserted = 0

with second_engine.begin() as destination:
    while True:
        batch = selected_rows.fetchmany(BATCH_SIZE)
        if not batch:
            break

        destination.execute(insert_statement, batch)
        rows_inserted += len(batch)

print(f"Inserted {rows_inserted} Rows.")

Inserted 39926 Rows.
